# Introduction

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data**<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    pool all data per location and year for time-dependent stationary analysis<br>

---
**Additional Notes**
- Using sim_year as the actual year 

**Notes on next steps:**
- 8 models × 680 locations = 5,440 independent analyses >> parallelize code on model level

---
**!!! ToDo**
- plot model fits
- create a mapping function for location info and lat/lon... store as JSON to save lookup
- add Confidence Intervals for Return Levels
- add return level (and CI) to txt
- store report output as parquet / Feather
- declutter what to save!
- time-series for stationary analysis (ignore shape and scale from global analysis)<br>
✓ check why slope $\mu(t)$ != $trend$ (m/year) -> mistake trend was calculated as $\mu(t)*years$


# Import Libraries

In [ ]:
import sys
import random
import time
from datetime import datetime
from glob import glob

import xarray as xr
from IPython.display import Markdown, display
from pandas import DataFrame

import func_gev as gev
import func_preparation as dbf
import func_utils as ut

# Settings

In [ ]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [ ]:
hindcast_start = 1960
hindcast_end = 2026

In [ ]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [ ]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

In [ ]:
print_msg = False
display_results = False
export_report=True

# Import data

In [ ]:
ls_files = [file for file in glob(path + '*.nc')]
ls_files

In [ ]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, using `joblib` - saving ~60% (from 3min30sec down to 1min22sec)


In [ ]:
time_start1 = datetime.now()
dic_data_per_model = dbf.data_preparation(ls_files=ls_files, dic_data_per_model=dic_data_per_model)
time_end1 = datetime.now()

In [ ]:
display(Markdown("**Data Overview**"))
display(Markdown("**Model · model shape: samples (~sim_years) | ensemble members | valid locations**"))
ut.create_data_overview(dic_data_per_model,ls_files)

display(Markdown(f"Execution time · {time_end1 - time_start1}sec"))

## Pool data per location cross models

from 8 models with up to 680 samples (sim_years) and two members and 3547-7216 locations, pool all data together and group per location
> Restructure multi-model ensemble by site <br>
> - from dic[model] -> shape: (sim_year, ensemble_member, location) <br>
> - to dic[location] -> shape: (sim_year, ensemble_member, model)


In [ ]:
da_list = []
for model_name, dic_model in dic_data_per_model.items():
    da = dic_model['valid data'] 

    da_loc = dbf.sites_to_location(da)
    da_loc = da_loc.expand_dims(model=[model_name])

    da_list.append(da_loc)

combined = xr.concat(da_list, dim="model", join="outer")

display(Markdown(f"\n**Overall, the combined dataset has the following dimensions**"))
print("Final dimensions:", combined.dims)
print("Shape:", combined.shape)
print("Number of models:", combined.model.size)
print("Number of locations:", combined.location.size)

### Validation Check

In [ ]:
list_model_labels = list(dic_data_per_model.keys())

In [ ]:
model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

display(Markdown(f"\n**Overview Original dataArray**"))
data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

display(Markdown(f"\n**Overview Revised dataArray**"))
revised_dataset, lon_rev, lat_rev = ut.get_dataset_overview_for_model_at_location(
    dic_data=combined, model_nr=model_ex, lon=lon_target, lat=lat_target
    )

In [ ]:
assert lon_target == lon_rev and lat_target == lat_rev
assert all(data_for_model_for_location.dropna() == revised_dataset.dropna())

**Learning** · make sure to never select by the site-id but always via geo-coordinates (lat | lon)

# Workflow GEV - Generalized Extreme Value

## OPTION1
Using all data available per location - from all years and models

#### Initial trial with subset of TWO locations
Later, batch(?) and parallelize

In [ ]:
list_sites = [42, 97, 50]

In [ ]:
columns_selected = ['sim_year', 'annualMax', 'lon', 'lat', 'member', 'model']

In [ ]:
dic_data_per_location = {}
for loc_ex in list_sites: 
    data_at_location = combined[:,:,:, loc_ex].to_dataframe().dropna().reset_index()#[columns_selected]
    data_at_location = data_at_location.rename(columns={'annualMax':'storm_surge'})
    dic_data_per_location[loc_ex] = data_at_location

### Run Analysis

In [ ]:
orig_stdout, orig_stderr, fh, logger, log_path = ut.initialize_logger(
    f"LOGS_GEVAnalysis_pooled_{datetime.now():%Y%m%d_%H%M%S}.log"
    )

# ------------------------------------------------------------------------------------------
time_start = time.time()

print("\n" + "="*100)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER LOCATION")
print("="*100)

# ------------------------------------------------------------------------------------------
dic_prepared = ut.prepare_pooled_data(
    dic_data=dic_data_per_location,
    hindcast_start=hindcast_start,
    hindcast_end=hindcast_end
)

# ------------------------------------------------------------------------------------------
results = {}
for loc_id, df_prepared in dic_prepared.items():
    print("\n" + "-"*70)
    print(f"Analyzing location id {loc_id} ...")

    lon_loc = df_prepared.lon.unique()[0]
    lat_loc = df_prepared.lat.unique()[0]

    print("\tLookup location info...")
    location_info = dbf.locations_label_lookup_simple(lon=lon_loc, lat=lat_loc)
    print(f"\t → Closest location identified: {location_info}")

    result = gev.analyze_per_location(
        df_prepared, lat_loc, lon_loc, location_info, return_periods
    )
    if result is None:
        print(f"\t→ Warning! No valid GEV fit for location id {loc_id}. Skipping ...")
        continue

    full_file_path = gev.create_gev_report_per_location(
        result_location=result,
        location_label=result['location info'][0],
        print_msg=print_msg,
        plot_period_evolution=plot_period_evolution,
        display_results=display_results,
        export_report=export_report,
        path_export=path_export,
    )
    result['file_path_report'] = full_file_path
    results[loc_id] = result

# ------------------------------------------------------------------------------------------
time_end = time.time()
print("\n" + "="*100)
print(f"✓ ANALYSIS COMPLETED IN {(time_end - time_start):.2f}s!")
print("="*100)

# ------------------------------------------------------------------------------------------
sys.stdout = orig_stdout
sys.stderr = orig_stderr
logger.removeHandler(fh)
fh.close()

Upscaling to 11022 locations, will result in an execution time of ~7.65hours!

## OPTION2
Re-run stationary GEV per year (for location parameter; scale and shape remain as globally defined)

In [ ]:
for site_id in results.keys():
    grp_per_year = dic_prepared[site_id].groupby('sim_year')

    print(f"Conducting stationary GEV for {site_id} and year...")

    results_stat_per_year_at_location = dict()
    for year, group in grp_per_year:
        print(f"\t...{int(year)}", end="\r") 

        data = (group
                .sort_values('sim_year')
                .reset_index(drop=True)
                .rename(columns={'sim_year': 'year', 'storm_surge': 'annual_max'})
                .dropna())
        annual_max_for_year = data['annual_max'].values
        
        gev_stationary = gev.fit_stationary_gev(annual_max_for_year)
        result_stat_per_year_at_location = dict({
            'location': gev_stationary['location'], 
            'n_obs': gev_stationary['n_obs'], 'log_likelihood': gev_stationary['log_likelihood'], 
            'aic': gev_stationary['aic'], 'bic': gev_stationary['bic'], 'dist_type': gev_stationary['dist_type'], 
            'tail_behavior': gev_stationary['tail_behavior']
            })
        results_stat_per_year_at_location[int(year)] = result_stat_per_year_at_location
    print("\nDone!\n") 

    results[site_id]['gev_stationary']['analysis_per_year'] = DataFrame.from_dict(results_stat_per_year_at_location).T
    
    gev.store_report_stationary_gev_per_year(results[site_id])


In [ ]:
from pandas import read_parquet

In [ ]:
file_meta = '../output/gev_analysis/pooled/2026-01-28/GEVanalysis_Portugal_32.704|-17.152_2026-01-28_metadata.parquet'
file_data = '../output/gev_analysis/pooled/2026-01-28/GEVanalysis_Portugal_32.704|-17.152_2026-01-28_data.parquet'
file_annual_analysis = '../output/gev_analysis/pooled/2026-01-28/GEVanalysis_Portugal_32.704|-17.152_2026-01-28_stat-gev_per_year.parquet'

In [ ]:
df_meta = read_parquet(file_meta)

df_annual_analysis_location = read_parquet(file_annual_analysis)
df_data_location = read_parquet(file_data)


---
to be continued

### Regression of location parameter over years
include uncertainty given by n_obs

In [ ]:
# Use this for regression (and include uncertainty of n_obs)
df_annual_analysis_location.location.plot(lw=0, marker='.', figsize=(10,3.5))